# 🧟 Downside-Up Complaint Bureau
## LAB 2 — NormalObjects Creative Complaint Handler (LangChain)

Welcome! This notebook builds **Becma**, a creative AI agent that handles complaints about the Normal Objects universe.

### What you'll learn
- How to give an AI agent **tools** it can call
- How the agent **picks which tools to use** on its own (no fixed order)
- How to track what the agent actually did

### How this notebook is organised
| Section | What happens |
|---|---|
| 1. Setup | Install packages, load API key |
| 2. Tools | Define the 4 functions Becma can call |
| 3. Agent | Build the agent with a system prompt |
| 4. Complaints | Feed in complaints and see the responses |
| 5. Analysis | Count which tools were used |

> **Tip:** Run cells top-to-bottom. Each cell builds on the one before it.

## Section 1 — Setup

First we install the required packages and load our OpenAI API key from the `.env` file.

In [ ]:
# Install required packages (only needs to run once)
# langchain        → the AI framework
# langchain-openai → connects LangChain to OpenAI's models
# python-dotenv    → reads the .env file so we don't hardcode secrets
%pip install langchain langchain-openai python-dotenv --quiet

In [ ]:
import os
import sys
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool          # decorator that turns a function into an agent tool
from langchain.agents import create_agent # builds the ReAct-style agent graph
from typing import List, Dict

# Fix Unicode output on Windows terminals
if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# Load OPENAI_API_KEY from the .env file in this folder
load_dotenv()

# Create the LLM (the AI brain)
# model        → GPT-4o-mini is fast and cheap, great for this
# temperature  → 0.7 means moderately creative (0 = robotic, 1 = chaotic)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print("Setup complete! LLM ready.")

## Section 2 — Tools

Tools are just normal Python functions with a `@tool` decorator.
The decorator tells LangChain: *"the agent is allowed to call this function."*

The **docstring** (the text in triple quotes) is what the AI reads to decide whether to use the tool — write it clearly!

We have 4 tools:
1. `consult_demogorgon` — chaotic monster perspective
2. `check_hawkins_records` — keyword search in a fake database
3. `cast_interdimensional_spell` — creative magical solutions
4. `gather_party_wisdom` — advice from the Stranger Things kids

In [ ]:
# ── TOOL 1 ──────────────────────────────────────────────────────────
# The Demogorgon gives a random, chaotic response.
# random.choice() picks one of 3 preset answers each time it's called.

@tool
def consult_demogorgon(complaint: str) -> str:
    """Get the Demogorgon's perspective on a complaint about the Upside Down.

    The Demogorgon is a creature from the Upside Down. It might have insights
    about interdimensional inconsistencies, but its perspective is... unique.

    Args:
        complaint: The complaint about the Upside Down

    Returns:
        The Demogorgon's perspective (creative and possibly chaotic)
    """
    responses = [
        f"The Demogorgon tilts its head. It seems confused by '{complaint}'. Perhaps the issue is that you're thinking in three dimensions?",
        f"The Demogorgon makes a sound that might be agreement. It suggests that the problem might be temporal - things work differently in the Upside Down's time.",
        f"The Demogorgon appears to be eating something. It doesn't seem to understand the concept of '{complaint}' - maybe consistency isn't a priority there?",
    ]
    return random.choice(responses)  # pick one at random


# Quick test — call the tool directly to see what it returns
print(consult_demogorgon.invoke("why are portals so unpredictable"))

In [ ]:
# ── TOOL 2 ──────────────────────────────────────────────────────────
# Searches a small hardcoded dictionary for matching keywords.
# If the query contains a known keyword (portal, monsters, etc.) it returns
# the matching record. Otherwise it returns a generic fallback.

@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins historical records for information.

    Hawkins, Indiana has a long history of strange occurrences. These records
    might contain clues about patterns or explanations.

    Args:
        query: What to search for in the records

    Returns:
        Information from Hawkins historical records
    """
    # Our fake "database" — key = keyword to match, value = record text
    records = {
        "portal":      "Records show portals have opened on various dates with no clear pattern. Weather, electromagnetic activity, and unknown factors seem involved.",
        "monsters":    "Historical records indicate creatures from the Upside Down behave differently based on environmental factors, time of day, and proximity to certain individuals.",
        "psychics":    "Records show that psychic abilities vary greatly. Some individuals can move objects but not see the future, others can see visions but not move things.",
        "electricity": "Hawkins has a history of electrical anomalies. Records suggest a connection between the Upside Down and electromagnetic fields.",
    }

    # Check if any keyword appears in the query (case-insensitive)
    for key, value in records.items():
        if key in query.lower():
            return value

    # Fallback if nothing matches
    return f"Records don't contain specific information about '{query}', but they note that many unexplained events have occurred in Hawkins over the years."


# Quick test
print(check_hawkins_records.invoke("tell me about electricity anomalies"))

In [ ]:
# ── TOOL 3 ──────────────────────────────────────────────────────────
# Picks 1, 2, or 3 random spells depending on the creativity_level.
# random.sample() picks N unique items from a list without repeating.

@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Suggest a creative interdimensional spell to fix a problem.

    Sometimes the best solution is a creative one that doesn't follow normal rules.
    This tool suggests imaginative fixes for Upside Down problems.

    Args:
        problem: The problem to solve
        creativity_level: How creative to be (low, medium, high)

    Returns:
        A creative spell or solution suggestion
    """
    # Map creativity level to number of spells to return
    creativity_multiplier = {"low": 1, "medium": 2, "high": 3}.get(creativity_level, 2)

    spells = [
        f"Try chanting 'Becma Becma Becma' three times while holding a Walkman. This might recalibrate the interdimensional frequencies related to: {problem}",
        f"Create a salt circle and place a compass in the center. The magnetic anomalies might help stabilize: {problem}",
        f"Play 'Running Up That Hill' backwards at the exact location of the issue. The temporal resonance could fix: {problem}",
        f"Gather three items: a lighter, a compass, and something personal. Arrange them in a triangle while thinking about: {problem}. The emotional connection might help.",
    ]

    # Pick N spells (no repeats), capped at however many exist
    selected = random.sample(spells, min(creativity_multiplier, len(spells)))
    return "\n".join(selected)


# Quick test — high creativity returns 3 spells
print(cast_interdimensional_spell.invoke({"problem": "portal inconsistency", "creativity_level": "high"}))

In [ ]:
# ── TOOL 4 ──────────────────────────────────────────────────────────
# Same keyword-matching pattern as check_hawkins_records,
# but returns dialogue from the Stranger Things kids instead of records.

@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the D&D party (Mike, Dustin, Lucas, Will) for their collective wisdom.

    The party has solved many mysteries together. Their combined knowledge
    and different perspectives can provide insights.

    Args:
        question: The question or problem to ask the party about

    Returns:
        The party's collective wisdom and suggestions
    """
    party_responses = {
        "portal":      "Mike: 'Portals are unpredictable, but they usually open near strong emotional events or electromagnetic disturbances.' Dustin: 'Also, they seem to follow some kind of pattern related to the Mind Flayer's activity.'",
        "monsters":    "Lucas: 'Demogorgons are territorial but also opportunistic.' Will: 'They can sense fear and strong emotions. Maybe that's why they act differently sometimes.'",
        "psychics":    "Mike: 'El's powers seem connected to her emotional state.' Dustin: 'And they're limited by her physical and mental energy. That's probably why she can't do everything.'",
        "electricity": "Lucas: 'The Upside Down seems to interfere with electrical systems.' Dustin: 'But it also creates strange connections. It's like a feedback loop.'",
    }

    for key, response in party_responses.items():
        if key in question.lower():
            return response

    # Generic fallback when no keyword matches
    return "The party huddles together. Mike: 'This is a tough one.' Dustin: 'We need more information.' Lucas: 'Let's think about what we know.' Will: 'Maybe we should consult other sources?'"


# Quick test
print(gather_party_wisdom.invoke("what do you know about psychics?"))

In [ ]:
# Put all 4 tools in a list — this is what we'll hand to the agent
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

print(f"Registered {len(tools)} tools:")
for t in tools:
    print(f"  • {t.name}")

## Section 3 — Build the Agent

An **agent** = LLM + tools + a system prompt that tells it how to behave.

`create_agent` builds a **ReAct loop** under the hood:
1. LLM reads the complaint
2. LLM decides which tool to call (or to stop)
3. Tool runs and returns a result
4. LLM reads the result and decides what to do next
5. Repeat until the LLM decides it has enough info to answer

The agent decides the tool order — we don't hardcode it.

In [ ]:
# The system prompt tells the AI who it is and how to behave.
# This is NOT the user's complaint — it's the background instructions.
SYSTEM_PROMPT = """You are Becma, the creative director of the Downside-Up Complaint Bureau.
Your job is to handle complaints about inconsistencies in the Normal Objects universe
(an alternate-reality version of Hawkins, Indiana).

You have access to several tools that give you information and creative solutions.
Use them freely and in any order you think is helpful. Be imaginative and entertaining
while still providing thoughtful explanations.

Guidelines:
- Use at least 2 different tools per complaint to cross-reference information
- Combine insights from multiple sources into a coherent (but fun) response
- Feel free to suggest creative, interdimensional solutions
- Keep responses entertaining but informative
- Always acknowledge the complaint before diving into solutions
"""

# create_agent wires the LLM and tools together into a runnable graph.
# system_prompt → the agent's personality/instructions
# tools         → the list of functions it can call
agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

print("Agent built! Becma is ready to handle complaints.")

## Section 4 — Handle Complaints

Now we feed complaints to the agent and watch it reason through them.

The `handle_complaint` function:
- Sends the complaint to the agent
- Prints each tool call as it happens
- Returns the final response

In [ ]:
def handle_complaint(complaint: str, tracker=None) -> str:
    """Send one complaint to the agent and display what happens step by step."""
    print(f"\n{'='*60}")
    print(f"COMPLAINT: {complaint}")
    print(f"{'='*60}\n")

    # invoke() sends the message and waits for the full response
    # The agent runs its ReAct loop internally before returning
    result = agent.invoke({"messages": [("human", complaint)]})

    # Walk through every message in the conversation history
    # to find the tool calls (type == "tool")
    tools_used = []
    for msg in result["messages"]:
        if getattr(msg, "type", "") == "tool":
            tool_name = getattr(msg, "name", "unknown")
            tools_used.append(tool_name)
            # Print the first 120 chars of the tool's output
            print(f"  [tool called: {tool_name}]")
            print(f"  → {str(msg.content)[:120]}...\n")

    # Tell the tracker which tools were used (for the analysis section)
    if tracker:
        tracker.record(tools_used)

    print(f"Tools used this complaint: {' → '.join(tools_used) if tools_used else 'none'}")

    # The last message in the list is always the final AI response
    return result["messages"][-1].content


print("handle_complaint() defined.")

In [ ]:
# The complaints we'll test with
complaints = [
    "Why do demogorgons sometimes eat people and sometimes don't?",
    "The portal opens on different days—is there a schedule?",
    "Why can some psychics see the Upside Down and others can't?",
    "Why do creatures and power lines react so strangely together?",
]

print(f"Ready to process {len(complaints)} complaints.")

In [ ]:
# Run complaint 1 — watch the tool calls appear
response = handle_complaint(complaints[0])
print(f"\n--- FINAL RESPONSE ---\n{response}")

In [ ]:
# Run complaint 2
response = handle_complaint(complaints[1])
print(f"\n--- FINAL RESPONSE ---\n{response}")

In [ ]:
# Run complaint 3
response = handle_complaint(complaints[2])
print(f"\n--- FINAL RESPONSE ---\n{response}")

In [ ]:
# Run complaint 4
response = handle_complaint(complaints[3])
print(f"\n--- FINAL RESPONSE ---\n{response}")

## Section 5 — Tool Usage Analysis

Now let's run all 4 complaints together and track which tools the agent picked.

The `ToolUsageTracker` is a simple counter — it records every tool name each time it's used.

In [ ]:
class ToolUsageTracker:
    """Counts how many times each tool was called across all complaints."""

    def __init__(self):
        # Start all counts at 0
        self.usage_count: Dict[str, int] = {t.name: 0 for t in tools}
        self.tool_sequences: List[str] = []  # full ordered list of every call

    def record(self, tool_names: List[str]):
        """Called after each complaint to log which tools were used."""
        for name in tool_names:
            if name in self.usage_count:
                self.usage_count[name] += 1
        self.tool_sequences.extend(tool_names)

    def get_statistics(self) -> Dict:
        return {
            "total_tool_calls": sum(self.usage_count.values()),
            "tool_counts": self.usage_count,
            # The tool with the highest count — or None if nothing was called
            "most_used": max(self.usage_count.items(), key=lambda x: x[1])[0]
            if any(self.usage_count.values()) else None,
            "full_sequence": self.tool_sequences,
        }


print("ToolUsageTracker defined.")

In [ ]:
# Run all 4 complaints and collect tool usage stats
tracker = ToolUsageTracker()

print("\n" + "="*60)
print("WELCOME TO THE DOWNSIDE-UP COMPLAINT BUREAU")
print("Becma's Chaos Mode — Activated")
print("="*60)

for complaint in complaints:
    response = handle_complaint(complaint, tracker)
    print(f"\n>>> FINAL RESPONSE:\n{response}\n")

In [ ]:
# Print the summary
stats = tracker.get_statistics()

print("\n" + "="*60)
print("TOOL USAGE SUMMARY")
print("="*60)
print(f"Total complaints  : {len(complaints)}")
print(f"Total tool calls  : {stats['total_tool_calls']}")
print(f"Most used tool    : {stats['most_used']}")
print()
print("Calls per tool:")
for tool_name, count in stats["tool_counts"].items():
    bar = "█" * count  # simple bar chart
    print(f"  {tool_name:<35} {bar} ({count})")
print()
print(f"Full sequence: {' → '.join(stats['full_sequence']) if stats['full_sequence'] else 'none recorded'}")

## Summary — What just happened?

You built a LangChain agent from scratch that:

1. **Reads a complaint** and decides what to do — no fixed script
2. **Calls tools in any order** it thinks is helpful
3. **Combines results** into a creative, entertaining response

### Key takeaway

| LangChain (this lab) | LangGraph (Lab 2 next week) |
|---|---|
| Agent picks tool order freely | You define the order in a graph |
| Great for open-ended creative tasks | Great for strict, auditable workflows |
| Less predictable | Fully predictable |

Use **LangChain** when creativity matters more than control.  
Use **LangGraph** when the order of steps must be guaranteed.